# Fine-Tuning a Large Language Model (LLM)

In previous experiments, we improved LLM performance using techniques such as **Multi-Shot Prompting**, **Tool usage**, and **Retrieval-Augmented Generation (RAG)**. These methods enhance responses **without changing the model’s internal weights**. In this section, we move beyond inference-time techniques and perform **Fine-Tuning**, where the model itself is trained further on a **custom dataset**.

Fine-tuning allows the LLM to learn domain-specific patterns, formats, or knowledge by updating its parameters. This can significantly improve performance for specialized tasks such as instruction following, domain Q&A, or structured outputs.

### Training Process

- **Training is the core idea of AI:** It involves setting the parameters of a model based on inputs and outputs.

- **Model Learning:** During training, the model processes the input text from the dataset and compares its predicted output with the correct output. The difference (loss) is calculated, and the model's internal weights are updated using backpropagation to improve future predictions.

- **Iterative Optimization:** This process is repeated over many batches and epochs. With each iteration, the model gradually learns patterns, language structure, and task-specific behavior from the dataset, improving its performance.

- **Inference on Unseen Data:** After training, the model is given new inputs (“Unseen Data”) to evaluate how well it performs.

**Generalization:**  
The ability of a model to make good predictions on unseen data is called **Generalization**.

*LLMs are remarkably good at generalization.*

### Training vs Fine-Tuning

#### Training (From Scratch)

- **Training is the core idea of AI:** It involves setting the parameters of a model based on inputs and outputs.
- During training, the model processes input data, compares predictions with the correct outputs, and updates its internal weights using backpropagation.
- This process repeats across many batches and epochs, gradually improving the model’s ability to learn patterns and relationships in the data.
- After training, the model is evaluated on **new inputs ("Unseen Data")** to test how well it performs.

**Generalization:**  
The ability of a model to make good predictions on unseen data is called **Generalization**.

*LLMs are remarkably good at generalization.*

---

#### Why We Use Fine-Tuning

- **Training a multi-billion parameter model from scratch is extremely expensive**, often costing **tens to hundreds of millions of dollars**.
- Instead, we take advantage of **Transfer Learning**.
- In this approach, we start with a **pretrained model** that already understands language patterns.
- We then **fine-tune the model using additional task-specific data** so it performs better for our particular use case.
- This makes development **much faster, cheaper, and more efficient** than training a model from scratch.

### Using a Custom Dataset from Hugging Face

To fine-tune the model, we will use a **custom dataset hosted on Hugging Face**. The dataset will be loaded using the Hugging Face `datasets` library and then used to train the model. During training, the model learns from the dataset examples and adapts its behavior accordingly.

The overall workflow includes:
1. Loading the dataset from Hugging Face.
2. Preprocessing and tokenizing the data.
3. Configuring the training parameters.
4. Training (fine-tuning) the model on the dataset.
5. Evaluating and saving the fine-tuned model.

This process enables the LLM to better align with the specific task or domain represented in the dataset.

### "THE PRICE IS RIGHT" Capstone Project

Now, we will build a model that predicts how much something costs from a description, based on a scrape of Amazon data.

1. Data Curation
2. Data Pre-processing
3. Evaluation, Baselines, Traditional ML
4. Deep Learning and LLMs
5. Fine-tuning a Frontier Model

### 1. Data Curation
Today we'll scrub our dataset and curate our data

The dataset is here:
[Amazon Review 2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023)

And the folder with all the product datasets is here:
[All Amazon Data 2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/tree/main/raw/meta_categories)



<span style="color:green;">

### Business value of Data Curation

Data Curation can be considered the less glamorous work of a Data Scientist. I say that's nonsense! This is where the science happens - what could be more glamorous than that?! R&D with your dataset can often have a greater impact on performance than the fashionable 'hyper-parameter optimization' that we do later. So: prepare for Quality Time with Data Quality.

</span>

In [ ]:
# Importing Libraries

import os
from huggingface_hub import login
from datasets import load_dataset
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import random
from pricer.items import Item
from pricer.parser import parse

In [ ]:
# Log in to HuggingFace

hf_token = os.environ['HUGGING_FACE_WRITE_TOKEN']
login(hf_token, add_to_git_credential=True)

This `Item` class is a **Pydantic data model** used to represent a **product with pricing information** and interact with the **Hugging Face Hub datasets**.

It stores product details, builds prompts for training models, and can **upload/download datasets**.

#### Load our dataset

In the next cell, we load in the dataset from huggingface.

If this gives you an error like "trust_remote_code is no longer supported", then please run this command in a new cell: !pip install --upgrade datasets==3.6.0 and then restart the Kernel, and try again.

Now we will use only one category for this from amazon dataset which is `Home Appliances` like Microwave, Dish Washer etc.

In [ ]:
dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Appliances", split="full", trust_remote_code=True)

In [ ]:
print(f"Number of Appliances: {len(dataset):,}")

In [ ]:
# Investigate a particular datapoint

dataset[5]

In [ ]:
# What's the most expensive item?

max_price = 0
max_item = None

for datapoint in tqdm(dataset):
    try:
        price = float(datapoint["price"])
        if price > max_price:
            max_item = datapoint
            max_price = price
    except ValueError:
        pass

print(f"The most expensive item is {max_item['title']} and it costs {max_price:,.2f}")

In [ ]:
# Load into Item objects if they have a price range $1-$1000 and enough details

items = [parse(datapoint, "Appliances") for datapoint in tqdm(dataset)]
items = [item for item in items if item is not None]
print(f"There are {len(items):,} items from {len(dataset):,} datapoints")

In [ ]:
items[0]

In [ ]:
print(items[0].full)

In [ ]:
prices = [item.price for item in items]
lengths = [len(item.full) for item in items]

In [ ]:
# Plot the distribution of lengths

plt.figure(figsize=(15,6))
plt.title(f"Lengths: Avg {sum(lengths)/len(lengths):,.0f} and highest {max(lengths):,}\n")
plt.xlabel('Length (chars)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color='lightblue', bins=range(0, 6000, 100))
plt.show()

In [ ]:
max_length = max(lengths)
max_length_item = items[lengths.index(max_length)]
print(max_length_item.full)

In [ ]:
# Plot the distribution of prices
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.2f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="orange", bins=range(0, 1000, 10))
plt.show()

In [ ]:
print(items[3].full)

In [ ]:
from pricer.loaders import ItemLoader
loader = ItemLoader("Appliances")
items = loader.load()

In [ ]:
dataset_names = [
    "Automotive",
    "Electronics",
    "Office_Products",
    "Tools_and_Home_Improvement",
    "Cell_Phones_and_Accessories",
    "Toys_and_Games",
    "Appliances",
    "Musical_Instruments",
]

In [ ]:
items = []
for dataset_name in dataset_names:
    loader = ItemLoader(dataset_name)
    items.extend(loader.load(workers=4))

In [ ]:
print(f"A grand total of {len(items):,} items")

In [ ]:
items[1000]

In [ ]:
# Removing Duplicates because same product appearing in 2 or more categories

random.seed(42)
random.shuffle(items)

seen = set()
items = [x for x in tqdm(items) if not (x.title in seen or seen.add(x.title))]

seen = set()
items = [x for x in tqdm(items) if not (x.full in seen or seen.add(x.full))]

del seen
print(f"After deduplication, we have {len(items):,} items")

In [ ]:
lengths = [len(item.full) for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Text length: Avg {sum(lengths)/len(lengths):,.1f} and highest {max(lengths):,}\n")
plt.xlabel('Length (characters)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="skyblue", bins=range(0, 4050, 50))
plt.show()

In [ ]:
# Plot the distribution of prices

prices = [item.price for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
from collections import Counter
category_counts = Counter([item.category for item in items])

categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')

for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')

plt.show()

In [ ]:
# Now the data set is heavily skewed towards cheap items and Automotives 
# This can affect our training results so we will select a subset of this dataset

# Now we will pick a subset but we will add weight to each category and select it 
# So that we will pick items in penalized way as it wont affect our subset

np.random.seed(42)

SIZE = 820_000

prices = np.array([it.price for it in items], dtype=float)
categories = np.array([it.category for it in items])
p = (prices - prices.min()) / (prices.max() - prices.min() + 1e-9)

w = p**2
w[categories == "Tools_and_Home_Improvement"] *= 0.5
w[categories == "Automotive"] *= 0.05

w = w / w.sum()
idx = np.random.choice(len(items), size=SIZE, replace=False, p=w)
sample = [items[i] for i in idx]

In [ ]:
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} lowest {min(prices):,} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
# Just for good measure, let's shuffle the sample again for the final dataset

random.seed(42)
random.shuffle(sample)

In [ ]:
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} lowest {min(prices):,} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
from collections import Counter
category_counts = Counter([item.category for item in sample])

categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

# Bar chart by category
plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')

plt.xticks(rotation=30, ha='right')

# Add value labels on top of each bar
for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')

# Display the chart
plt.show()

In [ ]:
# Automotive still in the lead, but improved somewhat
# For another perspective, let's look at a pie

plt.figure(figsize=(12, 10))
plt.pie(counts, labels=categories, autopct='%1.0f%%', startangle=90)

# Add a circle at the center to create a donut chart (optional)
centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)
plt.title('Categories')

# Equal aspect ratio ensures that pie is drawn as a circle
plt.axis('equal')  

plt.show()

In [ ]:
# How does the price vary with the character count?

sizes = [len(item.full) for item in sample]
prices = [item.price for item in sample]

# Create the scatter plot
plt.figure(figsize=(15, 8))
plt.scatter(sizes, prices, s=0.2, color="red")

# Add labels and title
plt.xlabel('Size')
plt.ylabel('Price')
plt.title('Is there a simple correlation with text length?')

# Display the plot
plt.show()


In [ ]:
# How does the price vary with the weight?

ounces = [item.weight for item in sample]
prices = [item.price for item in sample]

# Create the scatter plot
plt.figure(figsize=(15, 8))
plt.scatter(ounces, prices, s=0.2, color="darkorange")

# Add labels and title
plt.xlabel('Weight (ounces)')
plt.ylabel('Price')
plt.xlim(0, 400)
plt.title('Is there a simple correlation with weight?')

# Display the plot
plt.show()

### Pushing this dataset to Hugging Face

Now push this dataset to the HuggingFace Hub

Replace the username with your HF username if you've crafted your own dataset

Or, ignore this cell and you can load it from my repo!

In [ ]:
username = "Arivukkarasu"
full = f"{username}/Amazon_items_raw_full"
lite = f"{username}/Amazon_items_raw_lite"

train = sample[:800_000]
val = sample[800_000:810_000]
test = sample[810_000:]

Item.push_to_hub(full, train, val, test)

train_lite = train[:20_000]
val_lite = val[:1_000]
test_lite = test[:1_000]

Item.push_to_hub(lite, train_lite, val_lite, test_lite)